# Fluxo de Autorização — analise_siplan_rps

Acionado pelo **Power Automate** com uma lista de `atividade_id`. Atualiza a tabela
`analise_siplan_rps` e grava um arquivo JSON provisório no Lakehouse para retorno ao fluxo.

## O que faz

1. Parseia `atividade_ids_str` recebido do Power Automate.
2. Busca dados das atividades em `lake_gold_fatos.dbo.base`.
3. Registra/atualiza `wh_siplan_rps.dbo.analise_siplan_rps`:

| Condição | Ação |
|----------|------|
| `atividade_id` não existe | INSERT completo, Status via `get_status_inicial(autonomia)` |
| existe com Status em `STATUSES_ATUALIZAVEIS` | UPDATE (exceto `custos_foto`) |
| existe com outro Status | ignorado, registrado em log |

4. Grava `_output_{run_id}.json` no Lakehouse para leitura pelo Power Automate.

## Retorno (arquivo JSON no Lakehouse)

```json
{"gravado": true, "inseridos": 2, "atualizados": 1, "ignorados": 0, "run_id": "..."}
```


In [ ]:
atividade_ids_str   = ''   # ids passados pelo PA, separados por vírgula (ex: "123,456,789")
solicitante         = ''   # email de quem solicitou (injetado pelo PA)
run_id              = ''   # UUID gerado pelo PA; gerado internamente se vazio

ANALISE_TABLE = 'wh_siplan_rps.dbo.analise_siplan_rps'

SQL_ENDPOINT  = (
    'beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq'
    '.datawarehouse.fabric.microsoft.com'
)

# Lakehouse onde o arquivo de retorno é gravado
ONELAKE_WORKSPACE = 'ab5f231b-0d65-4a82-9fa3-e490336e912c'
ONELAKE_LAKEHOUSE = '59d32952-1672-4252-8c46-ff074684ed8e'
LAKEHOUSE_FOLDER  = 'Files/fluxo_entrega'

# ── Autonomias incluídas na saída (acoes_txt no JSON de retorno) ─────────────
AUTONOMIAS_SAIDA = frozenset({'DIREG', 'STS'})

# ── Mapeamento autonomia → Status no INSERT ──────────────────────────────────
# Edite este dicionário para adicionar/remover regras sem mexer no código.
STATUS_POR_AUTONOMIA = {
    'UO': 'AutonomiaUO',   # autonomia da UO → não vai para STS
}
STATUS_PADRAO = 'Enviada'  # Status padrão quando autonomia não está mapeada

# ── Status que permitem UPDATE quando a linha já existe ──────────────────────
# Linhas com outros Status são ignoradas (log apenas).
STATUSES_ATUALIZAVEIS = frozenset({
    'Enviada',      # entrega inicial registrada
    'Reenviada',    # correção de entrega anterior
    'Em revisão',   # aguardando parecer
    'AutonomiaUO',  # autonomia da UO — pode ser reprocessado
})


In [ ]:
# ── Imports e detecção de ambiente ───────────────────────────────────────────
import json
import struct
import uuid
import warnings
from datetime import datetime, timedelta

import pandas as pd

warnings.filterwarnings('ignore')

try:
    spark
    FABRIC_ENV = True
    print('Ambiente: Microsoft Fabric')
except NameError:
    FABRIC_ENV = False
    print('Ambiente: local')

try:
    from notebookutils import mssparkutils as _ms
    mssparkutils = _ms
    HAS_MSSPARKUTILS = True
except ImportError:
    HAS_MSSPARKUTILS = False

try:
    import pyodbc
except ImportError:
    print('[AVISO] pyodbc não disponível — instale via pip install pyodbc')
    raise

print(f'Imports OK — {datetime.now():%d/%m/%Y %H:%M}')


In [ ]:
# ── Conexão com o SQL endpoint ───────────────────────────────────────────────

def _get_conn():
    """pyodbc com token Azure AD — uso local apenas."""
    from azure.identity import InteractiveBrowserCredential, DeviceCodeCredential
    try:
        cred = InteractiveBrowserCredential()
    except Exception:
        cred = DeviceCodeCredential()
    token = cred.get_token('https://database.windows.net/.default').token
    tb = token.encode('utf-16-le')
    ts = struct.pack(f'<I{len(tb)}s', len(tb), tb)
    return pyodbc.connect(
        f'DRIVER={{ODBC Driver 17 for SQL Server}};'
        f'SERVER={SQL_ENDPOINT};Encrypt=Yes;',
        attrs_before={1256: ts},
    )


def _get_jdbc_conn():
    """JDBC via JVM com token mssparkutils — uso Fabric apenas."""
    token = mssparkutils.credentials.getToken('https://database.windows.net/')
    ds = spark._jvm.com.microsoft.sqlserver.jdbc.SQLServerDataSource()
    ds.setServerName(SQL_ENDPOINT)
    ds.setPortNumber(1433)
    ds.setEncrypt(True)
    ds.setAccessToken(token)
    return ds.getConnection()


print('Auth helpers definidos.')


In [ ]:
# ── Busca dados das atividades com todos os campos necessários ────────────────
def fetch_atividades(ids: list) -> pd.DataFrame:
    ids_sql = ', '.join(str(int(i)) for i in ids)

    # Fabric: spark.sql com CAST AS STRING e JOIN cross-lakehouse
    sql_fabric = (
        'SELECT '
        '    b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '    b.PrimeiraData AS dataPrimeiraSessao, '
        '    b.areaprog AS area, '
        '    b.linguagem, b.mes, b.autonomia, '
        '    b.complemento, b.item_desc, '
        '    b.projeto_nome AS projeto, '
        '    b.precificacao_desc, '
        '    du.unidade, '
        '    MAX(ds.localNome) AS localNome_max '
        'FROM lake_gold_fatos.dbo.base b '
        'LEFT JOIN lake_gold_fatos.dbo.dim_unidade du '
        '       ON CAST(LEFT(CAST(CAST(b.atividade_id AS BIGINT) AS STRING), 2) AS INT) = du.uo '
        'LEFT JOIN lake_gold_fatos.dbo.datas_sessoes ds '
        '       ON b.atividade_id = ds.atividade_id '
        f'WHERE b.atividade_id IN ({ids_sql}) '
        'GROUP BY b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '         b.PrimeiraData, b.areaprog, b.linguagem, b.mes, b.autonomia, '
        '         b.complemento, b.item_desc, b.projeto_nome, b.precificacao_desc, '
        '         du.unidade'
    )

    # Local: pyodbc com CAST AS VARCHAR e subquery para localNome
    sql_local = (
        'SELECT '
        '    b.atividade_id, b.nome, b.custo_total, b.gerencia, '
        '    b.PrimeiraData AS dataPrimeiraSessao, '
        '    b.areaprog AS area, '
        '    b.linguagem, b.mes, b.autonomia, '
        '    b.complemento, b.item_desc, '
        '    b.projeto_nome AS projeto, '
        '    b.precificacao_desc, '
        '    du.unidade, '
        '    (SELECT MAX(ds.localNome) '
        '       FROM lake_gold_fatos.dbo.datas_sessoes ds '
        '      WHERE ds.atividade_id = b.atividade_id) AS localNome_max '
        'FROM lake_gold_fatos.dbo.base b '
        'LEFT JOIN lake_gold_fatos.dbo.dim_unidade du '
        '       ON CAST(LEFT(CAST(b.atividade_id AS VARCHAR(20)), 2) AS INT) = du.uo '
        f'WHERE b.atividade_id IN ({ids_sql})'
    )

    if FABRIC_ENV:
        df = spark.sql(sql_fabric).toPandas()
    else:
        conn = _get_db_conn()
        df   = pd.read_sql(sql_local, conn)
        conn.close()

    df['atividade_id'] = pd.to_numeric(df['atividade_id'], errors='coerce').astype('Int64')
    df['custo_total']  = pd.to_numeric(df['custo_total'],  errors='coerce').fillna(0.0)
    return df


print('fetch_atividades definida.')

In [ ]:
# ── Helpers: construção de campos e execução de DML ──────────────────────────

_DIAS_PT = ['seg', 'ter', 'qua', 'qui', 'sex', 'sáb', 'dom']


def _fmt_data_custos(v) -> str:
    dt = pd.to_datetime(v, errors='coerce')
    if pd.isna(dt):
        return str(v or '')
    return f'{_DIAS_PT[dt.weekday()]}, {dt.day:02d}/{dt.month:02d}/{dt.year}'


def _fmt_mes(mes, data_primeira) -> str:
    dt = pd.to_datetime(data_primeira, errors='coerce')
    if pd.isna(dt) or not mes:
        return str(mes or '')
    return f'{dt.year}-{str(mes).strip()}'


def _build_custos(row) -> str:
    """Texto multilinha para custos_foto e custos_editavel."""
    local_prec = (
        str(row.get('localNome_max') or '') + ' - ' + str(row.get('precificacao_desc') or '')
    ).strip(' -')
    partes = [
        _fmt_data_custos(row.get('dataPrimeiraSessao')),
        str(row.get('complemento') or ''),
        str(row.get('item_desc') or ''),
        str(row.get('projeto') or ''),
        local_prec,
    ]
    return chr(10).join(p for p in partes if p)


def _quote_col(c: str) -> str:
    return f'[{c}]'


def get_status_inicial(autonomia) -> str:
    """Retorna o Status a usar no INSERT com base na autonomia da atividade."""
    return STATUS_POR_AUTONOMIA.get(str(autonomia or '').strip(), STATUS_PADRAO)


def _sql_val(v) -> str:
    """Serializa valor Python para literal T-SQL seguro (sem bind params)."""
    if v is None:
        return 'NULL'
    if isinstance(v, bool):
        return '1' if v else '0'
    if isinstance(v, (int, float)):
        return str(v)
    if isinstance(v, datetime):
        return chr(39) + v.strftime('%Y-%m-%d %H:%M:%S') + chr(39)
    s = str(v).replace(chr(39), chr(39) + chr(39))   # escapa aspas simples
    return chr(39) + s + chr(39)


def _exec_sql(sql: str) -> None:
    """DML no warehouse: JDBC no Fabric, pyodbc local."""
    if FABRIC_ENV:
        conn = _get_jdbc_conn()
        stmt = conn.createStatement()
        stmt.execute(sql)
        conn.close()
    else:
        conn = _get_conn()
        conn.execute(sql)
        conn.commit()
        conn.close()


def _query_existentes(ids_sql: str) -> pd.DataFrame:
    sel = (
        f'SELECT atividade_id, Status '
        f'FROM {ANALISE_TABLE} '
        f'WHERE atividade_id IN ({ids_sql})'
    )
    if FABRIC_ENV:
        return spark.sql(sel).toPandas()
    conn = _get_conn()
    df   = pd.read_sql(sel, conn)
    conn.close()
    return df


print('Helpers definidos.')


In [ ]:
# ── Lógica principal de INSERT / UPDATE ──────────────────────────────────────
def registrar_analise(df_ativ: pd.DataFrame,
                      data_entrega: datetime, quem: str) -> dict:
    """
    Para cada atividade_id:
      - Não existe → INSERT completo (Status via get_status_inicial)
      - Existe com Status em STATUSES_ATUALIZAVEIS
            → UPDATE campos de entrega + dados da base (exceto custos_foto)
      - Existe com outro Status → ignora, registra em log

    Retorna dict com contagens e acoes_txt (linhas DIREG/STS formatadas).
    """
    ids = df_ativ['atividade_id'].dropna().astype('int64').tolist()
    if not ids:
        print('Nenhum atividade_id para processar.')
        return {'inseridos': 0, 'atualizados': 0, 'ignorados': 0, 'acoes_txt': ''}

    ids_sql      = ', '.join(str(i) for i in ids)
    existentes   = _query_existentes(ids_sql)
    status_atual = dict(zip(
        existentes['atividade_id'].astype('int64'),
        existentes['Status'],
    ))

    novos, atualizados, ignorados, acoes_detalhes = [], [], [], []

    for _, row in df_ativ.iterrows():
        aid    = int(row['atividade_id'])
        custos = _build_custos(row)
        aut    = str(row.get('autonomia') or '').strip()

        if aid not in status_atual:
            status_final = get_status_inicial(aut)
            novos.append({
                'id':                 int(uuid.uuid4().hex[:15], 16),
                'atividade_id':       aid,
                'nome':               row.get('nome') or '',
                'Título':             row.get('nome') or '',
                'unidade':            row.get('unidade') or '',
                'gerencia':           row.get('gerencia') or '',
                'area':               row.get('area') or '',
                'linguagem':          row.get('linguagem') or '',
                'mes':                _fmt_mes(row.get('mes'), row.get('dataPrimeiraSessao')),
                'autonomia':          aut,
                'dataPrimeiraSessao': row.get('dataPrimeiraSessao'),
                'custos_foto':        custos,
                'custos_editavel':    custos,
                'data_entrega':       data_entrega,
                'quem':               quem,
                'Criado por':         quem,
                'Criado':             data_entrega,
                'Status':             status_final,
            })

        elif status_atual[aid] in STATUSES_ATUALIZAVEIS:
            status_final = 'Reenviada'
            atualizados.append({
                'atividade_id':       aid,
                'nome':               row.get('nome') or '',
                'Título':             row.get('nome') or '',
                'unidade':            row.get('unidade') or '',
                'gerencia':           row.get('gerencia') or '',
                'area':               row.get('area') or '',
                'linguagem':          row.get('linguagem') or '',
                'mes':                _fmt_mes(row.get('mes'), row.get('dataPrimeiraSessao')),
                'autonomia':          aut,
                'dataPrimeiraSessao': row.get('dataPrimeiraSessao'),
                'custos_editavel':    custos,
                'data_entrega':       data_entrega,
                'quem':               quem,
                'Status':             status_final,
                'Modificado':         data_entrega,
                'Modificado por':     quem,
            })

        else:
            status_final = status_atual[aid]
            ignorados.append((aid, status_final))

        if aut in AUTONOMIAS_SAIDA:
            acoes_detalhes.append({
                '_aid':               aid,
                'autonomia':          aut,
                'dataPrimeiraSessao': row.get('dataPrimeiraSessao'),
                'nome':               row.get('nome') or '',
                'gerencia':           row.get('gerencia') or '',
                'area':               row.get('area') or '',
                'Status':             status_final,
            })

    # INSERT
    for item in novos:
        cols = ', '.join(_quote_col(c) for c in item)
        vals = ', '.join(_sql_val(v) for v in item.values())
        _exec_sql(f'INSERT INTO {ANALISE_TABLE} ({cols}) VALUES ({vals})')

    # UPDATE
    for item in atualizados:
        aid  = item.pop('atividade_id')
        sets = ', '.join(f'{_quote_col(c)} = {_sql_val(v)}' for c, v in item.items())
        _exec_sql(
            f'UPDATE {ANALISE_TABLE} '
            f'SET {sets} '
            f'WHERE atividade_id = {aid}'
        )

    # Formata saída das ações DIREG/STS: ordena por autonomia e data
    def _sort_key(a):
        dt = pd.to_datetime(a['dataPrimeiraSessao'], errors='coerce')
        return (a['autonomia'], '' if pd.isna(dt) else dt.isoformat())

    def _fmt_linha(a):
        dt = pd.to_datetime(a['dataPrimeiraSessao'], errors='coerce')
        dt_str = (
            f"{_DIAS_PT[dt.weekday()]}, {dt.day:02d}/{dt.month:02d}/{dt.year}"
            if pd.notna(dt) else str(a['dataPrimeiraSessao'] or '')
        )
        return (f"{a['autonomia']} - {dt_str} - {a['nome']} - "
                f"{a['gerencia']} - {a['area']} - {a['Status']}")

    ignorados_ids = {a for a, _ in ignorados}
    processadas = sorted([a for a in acoes_detalhes if a['_aid'] not in ignorados_ids], key=_sort_key)
    nao_proc    = sorted([a for a in acoes_detalhes if a['_aid']     in ignorados_ids], key=_sort_key)
    linhas = [_fmt_linha(a) for a in processadas]
    if nao_proc:
        linhas += ['', 'Não processadas'] + [_fmt_linha(a) for a in nao_proc]
    acoes_txt = '\n'.join(linhas)

    counts = {
        'inseridos':   len(novos),
        'atualizados': len(atualizados),
        'ignorados':   len(ignorados),
        'acoes_txt':   acoes_txt,
    }
    print(f'analise_siplan_rps: inseridos={len(novos)}, atualizados={len(atualizados)}, ignorados={len(ignorados)}')
    for aid, st in ignorados:
        print(f'  atividade {aid} ignorada (Status={st!r})')
    return counts


print('registrar_analise definida.')


In [ ]:
# ── Execução ──────────────────────────────────────────────────────────────────

# 1. Parseia IDs
raw = atividade_ids_str.strip()
if raw.startswith('{'):
    raw = '[' + raw + ']'
if raw.startswith('['):
    parsed = json.loads(raw)
    if parsed and isinstance(parsed[0], dict):
        ids = [int(x['atividade_id']) for x in parsed]
    else:
        ids = [int(x) for x in parsed]
else:
    ids = [int(x.strip()) for x in raw.split(',') if x.strip()]

print(f'{len(ids)} atividades: {ids}')

# 2. Busca dados e grava analise_siplan_rps
df     = fetch_atividades(ids)
print(f'Dados carregados: {len(df)} linha(s)')
now_brt = datetime.utcnow() + timedelta(hours=-3)
counts = registrar_analise(df, now_brt, solicitante)

# 3. UUID do run — usa o injetado pelo PA; garante isolamento por execução
rid = run_id.strip() if run_id.strip() else str(uuid.uuid4())

# 4. Monta resultado e grava _output_{rid}.json no Lakehouse
saida = json.dumps(
    {'gravado': True, 'run_id': rid, **counts},
    ensure_ascii=False,
)

if HAS_MSSPARKUTILS:
    output_json_name = f'_output_{rid}.json'
    abfss_json = (
        f'abfss://{ONELAKE_WORKSPACE}'
        f'@onelake.dfs.fabric.microsoft.com'
        f'/{ONELAKE_LAKEHOUSE}/{LAKEHOUSE_FOLDER}/{output_json_name}'
    )
    mssparkutils.fs.put(abfss_json, saida, overwrite=True)
    print(f'{output_json_name} gravado em {LAKEHOUSE_FOLDER}/')

print(saida)

if HAS_MSSPARKUTILS:
    mssparkutils.notebook.exit(saida)
